In [ ]:
from pyspark.sql.functions import (
    col, trim, when, max as spark_max, explode, lit, from_json, to_date, lower
)

from delta.tables import DeltaTable

tabela_nome = "silver_extratos"

if spark.catalog.tableExists("controle_pipeline"):
        df_controle = spark.table("controle_pipeline").filter(col("tabela") == tabela_nome)

        if df_controle.count() > 0: 
            ultimo_processamento = df_controle.select("ultimo_ts").collect()[0][0]
        else:
            ultimo_processamento = None
else: 
    ultimo_processamento = None

In [ ]:
nome_tabela = "extrato_bancario"
caminho_bronze = f"Files/bronze/{nome_tabela}"
df_bronze = spark.read.format("delta").load(caminho_bronze)

if ultimo_processamento:
    df_bronze = df_bronze.filter(
        col("ingestion_ts") > ultimo_processamento
    )

if df_bronze.rdd.isEmpty():
    print("Sem dados novos para processar.")
    dbutils.notebook.exit("No data")

In [ ]:
display(df_bronze)

In [ ]:
from pyspark.sql.types import *

schema = StructType([
    StructField("cCodCategoria", StringType()),
    StructField("cDataInclusao", StringType()),
    StructField("cDesCategoria", StringType()),
    StructField("cDesCliente", StringType()),
    StructField("cDocCliente", StringType()),
    StructField("cDocumentoFiscal", StringType()),
    StructField("cHoraInclusao", StringType()),
    StructField("cNatureza", StringType()),
    StructField("cOrigem", StringType()),
    StructField("cParcela", StringType()),
    StructField("cRazCliente", StringType()),
    StructField("cSituacao", StringType()),
    StructField("cTipoDocumento", StringType()),
    StructField("dDataConciliacao", StringType()),
    StructField("dDataLancamento", StringType()),

    StructField("nCodCliente", LongType()),
    StructField("nCodLancRelac", LongType()),
    StructField("nCodLancamento", LongType()),

    StructField("nSaldo", DecimalType(18, 2)),
    StructField("nValorDocumento", DecimalType(18, 2)),

    StructField("empresa", StringType()),
    StructField("conta", LongType())

])

df_bronze = df_bronze.withColumn("json", from_json(col("raw_json"), schema))
df_silver = df_bronze.select("json.*", "ingestion_ts")

In [ ]:
display(df_silver)

In [ ]:
for c, t in df_silver.dtypes:
    if t == "string":
        df_silver = df_silver.withColumn(
            c,
            when(trim(col(c)) == "", None)
            .otherwise(trim(col(c)))
        )

In [ ]:
# adicionar esse bloco nas tabelas silver já montadas
df_silver = df_silver.withColumn("cDataInclusao", to_date(col("cDataInclusao"), "dd/MM/yyyy")) \
            .withColumn("dDataConciliacao", to_date(col("dDataConciliacao"), "dd/MM/yyyy")) \
            .withColumn("dDataLancamento", to_date(col("dDataLancamento"), "dd/MM/yyyy"))

In [ ]:
df_silver = df_silver.withColumn("tipo_linha", when(lower(trim(col("cDesCliente"))).startswith("saldo"), "saldo")
                                                .otherwise("movimentacao")
)

In [ ]:
display(df_silver)

In [ ]:

df_silver = df_silver.dropDuplicates([
    "nCodLancamento",
    "dDataLancamento",
    "nSaldo",
    "empresa",
    "conta"
])

In [ ]:
if spark.catalog.tableExists("silver_extratos"):

    delta_table = DeltaTable.forName(spark, "silver_extratos")

    (delta_table.alias("t")
            .merge(
                df_silver.alias("s"),
                """
                t.nCodLancamento = s.nCodLancamento
                AND t.empresa = s.empresa
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()  
    )

else: 
    df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_extratos")

In [ ]:
novo_max_ts = df_bronze.agg(
    spark_max("ingestion_ts").alias("max_ts")).collect()[0]["max_ts"]

df_update = spark.createDataFrame(
    [(tabela_nome, novo_max_ts)],
    ["tabela", "ultimo_ts"]
)

if spark.catalog.tableExists("controle_pipeline"):

    delta_ctrl = DeltaTable.forName(spark, "controle_pipeline")

    (delta_ctrl.alias("t")
        .merge(
            df_update.alias("s"),
            "t.tabela = s.tabela"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else: 
    df_update.write.format("delta").mode("overwrite").saveAsTable("controle_pipeline")